In [1]:
!pip install underthesea tokenizers -q 
# !pip install pandarallel -q 
!pip install pyarrow -q 
!pip install evaluate rouge_score -q
!pip install torchinfo -q 
!pip install pyngrok -q
!pip install bert_score -q
!pip install google-genai pydantic pandas -q 

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.3/7.3 MB 65.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 56.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 2.4 MB/s eta 0:00:00


In [2]:
from underthesea import word_tokenize
import pandas as pd
import os
import random
import math
import copy
import multiprocessing
import time
import subprocess
import numpy as np
from tqdm import tqdm
import gc
import json

# --- THƯ VIỆN GEMINI NATIVE NỀN TẢNG MỚI ---
from google import genai
from google.genai import types
from pydantic import BaseModel, Field
from typing import List
# ---------------------------------------------

import torch
from pyngrok import ngrok
from kaggle_secrets import UserSecretsClient
from IPython.display import display

def seed(seed_value=42):
    os.environ['PYTHONHASHSEED'] = str(seed_value)
    torch.manual_seed(seed_value)
    random.seed(seed_value)
    np.random.seed(seed_value)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed_value)
        torch.cuda.manual_seed_all(seed_value)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
    print('Done')

def seed_worker(worker_id):
    worker_seed = torch.initial_seed() % 2**32
    np.random.seed(worker_seed)
    random.seed(worker_seed)

# =====================================================================
# CẤU HÌNH SDK GEMINI CHUẨN ĐÃ FIX LỖI
# =====================================================================
user_secrets = UserSecretsClient()
GEMINI_API_KEY = user_secrets.get_secret("GEMINI_API_KEY") 

client = genai.Client(api_key=GEMINI_API_KEY)
MODEL_NAME = "gemini-3.1-flash-lite" 

# ĐỊNH NGHĨA PYDANTIC SCHEMA - Viết hoa khớp chuẩn với Prompt của bạn
class GEvalSchema(BaseModel):
    Relevance: List[int] = Field(description="Scores from 0 to 100 for the 4 summaries in order")
    Coherence: List[int] = Field(description="Scores from 0 to 100 for the 4 summaries in order")
    Consistency: List[int] = Field(description="Scores from 0 to 100 for the 4 summaries in order")
    Fluency: List[int] = Field(description="Scores from 0 to 100 for the 4 summaries in order")
    Entity_Precision: List[int] = Field(description="Scores from 0 to 100 for the 4 summaries in order")

# Đọc file dữ liệu Master CSV 
df_master_saved = pd.read_csv('/kaggle/input/datasets/longnguyen2k5/test-set-prediction-results/master_evaluation_dataset.csv')
model_columns = [col for col in df_master_saved.columns if col not in ['source', 'reference']]

EVALUATION_PROMPT_TEMPLATE = """
Bạn sẽ được cung cấp một bài báo gốc và 4 bản tóm tắt của nó. Nhiệm vụ của bạn là đánh giá 4 bản tóm tắt này dựa trên 5 tiêu chí khác nhau.
Vui lòng đọc và hiểu kỹ các hướng dẫn này.

Tiêu chí và Các bước đánh giá:

1. Relevance (Sự liên quan).
Tiêu chí: Bản tóm tắt chỉ nên bao gồm những thông tin quan trọng từ tài liệu gốc. Hãy trừ điểm đối với các bản tóm tắt chứa thông tin dư thừa, lặp lại và không cần thiết.
Các bước đánh giá:
1. Đọc kỹ bản tóm tắt và tài liệu gốc.
2. So sánh bản tóm tắt với tài liệu gốc và xác định các ý chính của bài báo.
3. Đánh giá mức độ bản tóm tắt bao quát các ý chính của bài báo, và mức độ chứa thông tin dư thừa hoặc không liên quan.
4. Chấm điểm sự liên quan bằng một số nguyên từ 0 đến 100.

2. Coherence (Tính mạch lạc).
Tiêu chí: Bản tóm tắt phải được cấu trúc và tổ chức tốt. Bản tóm tắt không nên chỉ là một tập hợp các thông tin rời rạc, mà phải được liên kết từ câu này sang câu khác thành một khối thông tin mạch lạc về chủ đề.
Các bước đánh giá:
1. Đọc kỹ bài báo và xác định chủ đề chính cùng các ý chính.
2. Đọc bản tóm tắt và so sánh với bài báo. Kiểm tra xem bản tóm tắt có bao quát chủ đề chính và các ý chính của bài báo hay không, và liệu nó có trình bày chúng theo một trình tự rõ ràng và hợp lý không.
3. Chấm điểm tính mạch lạc theo thang điểm từ 0 đến 100, trong đó 0 là thấp nhất và 100 là cao nhất dựa trên Tiêu chí đánh giá.

3. Consistency (Tính nhất quán).
Tiêu chí: Một bản tóm tắt nhất quán về mặt thực tế chỉ chứa các câu khẳng định được hỗ trợ và suy ra từ tài liệu gốc. Hãy trừ điểm nặng đối với các bản tóm tắt chứa thông tin bịa đặt, sai lệch, hoặc không có trong bài gốc.
Các bước đánh giá:
1. Đọc kỹ bài báo và xác định các sự kiện và chi tiết chính mà nó trình bày.
2. Đọc bản tóm tắt và so sánh với bài báo. Kiểm tra xem bản tóm tắt có chứa bất kỳ lỗi sai sự thật hoặc thông tin nào không được bài báo hỗ trợ hay không.
3. Chấm điểm tính nhất quán từ 0 đến 100 dựa trên Tiêu chí đánh giá.

4. Fluency (Tính trôi chảy).
Tiêu chí: Chất lượng của bản tóm tắt về mặt ngữ pháp, chính tả, dấu câu, cách chọn từ và cấu trúc câu.
- 0-33: Kém. Bản tóm tắt có nhiều lỗi khiến văn bản khó hiểu hoặc đọc nghe rất thiếu tự nhiên.
- 34-66: Khá. Bản tóm tắt có some lỗi ảnh hưởng đến sự rõ ràng hoặc trôi chảy của văn bản, nhưng các ý chính vẫn có thể hiểu được.
- 67-100: Tốt. Bản tóm tắt có rất ít hoặc không có lỗi, hành văn tự nhiên, dễ đọc và dễ theo dõi.
Các bước đánh giá: Đọc bản tóm tắt và đánh giá tính trôi chảy của nó dựa trên các tiêu chí đã cho. Chấm điểm tính trôi chảy bằng một số nguyên từ 0 đến 100.

5. Entity_Precision (Độ chính xác thực thể).
Tiêu chí: Đánh giá mức độ bảo toàn và tính chính xác tuyệt đối của các thực thể cốt lõi (Tên riêng, Tên người, Địa danh, Tên tổ chức, Thương hiệu, Giải thưởng, Số liệu) xuất hiện trong bản tóm tắt so với tài liệu gốc. Tiêu chí này dùng để phát hiện lỗi ảo giác thực thể (Entity Hallucination).
- 0-33: Kém. Bản tóm tắt chứa các thực thể hoàn toàn bịa đặt (không hề có trong bài gốc), hoặc làm biến đổi nghiêm trọng thông tin thực thể, xảy ra lỗi "râu ông nọ cắm cằm bà kia" (Ví dụ bài gốc viết về "InterContinental Phú Quốc" nhưng bản tóm tắt lại viết thành "InterContinental Đà Nẵng" hoặc "Fleur de Lys Hotel").
- 34-66: Khá. Bản tóm tắt trích xuất được một số thực thể đúng, nhưng có xu hướng bỏ sót các thực thể quan trọng hoặc thay thế chúng bằng danh từ chung quá trừu tượng làm mờ nhạt bản sắc ngữ nghĩa (Ví dụ đổi "InterContinental Phu Quoc Long Beach Resort" thành "Một khu nghỉ dưỡng gia đình").
- 67-100: Tốt. Bản tóm tắt bảo toàn nguyên vẹn và chuẩn xác danh tính từ vựng của các tên riêng, địa danh cốt lõi. Không bịa đặt, không nhầm lẫn đối tượng, hành văn trích xuất thông tin thực thể sắc bén và đáng tin cậy.
Các bước đánh giá:
1. Xác định và lập danh sách các thực thể (Tên riêng, địa danh, số liệu...) xuất hiện trong bản tóm tắt.
2. Đối chiếu chéo từng thực thể đó với văn bản gốc. Kiểm tra xem có thực thể nào bịa đặt hoặc bị lắp ghép sai lệch ngữ cảnh hay không.
3. Chấm điểm độ chính xác thực thể bằng một số nguyên từ 0 đến 100 dựa trên các phân cấp đã cho.

Dữ liệu:
Văn bản gốc:
{document}

Bản tóm tắt 1:
{summary1}

Bản tóm tắt 2:
{summary2}

Bản tóm tắt 3:
{summary3}

Bản tóm tắt 4:
{summary4}

Đánh giá:
Chỉ cung cấp ĐIỂM SỐ là các số nguyên từ 0 đến 100 cho mỗi tiêu chí, tương ứng với thứ tự của các bản tóm tắt.
Định dạng đầu ra bắt buộc phải là một đối tượng JSON hợp lệ có chứa 5 mảng điểm số cho 'Relevance', 'Coherence', 'Consistency', 'Fluency', và 'Entity_Precision'.
"""

# =====================================================================
# HÀM KẾT NỐI API GEMINI NATIVE (ĐÃ FIX THAM SỐ)
# =====================================================================
def call_geval_judge(document, summaries):
    prompt = EVALUATION_PROMPT_TEMPLATE.format(
        document=document,
        summary1=summaries[0],
        summary2=summaries[1],
        summary3=summaries[2],
        summary4=summaries[3]
    )
    
    try:
        # Đã sửa đổi tham số cấu hình chuẩn xác thành `response_schema`
        response = client.models.generate_content(
            model=MODEL_NAME,
            contents=prompt,
            config=types.GenerateContentConfig(
                response_mime_type="application/json",
                response_schema=GEvalSchema, 
                temperature=0.0                  
            )
        )
        return json.loads(response.text)
    except Exception as e:
        # Trả về mảng rỗng tạm thời nếu có lỗi kết nối thực tế xảy ra để không dừng pipeline
        return {m.lower(): [50, 50, 50, 50] for m in ['relevance', 'coherence', 'consistency', 'fluency', 'entity_precision']}

# =====================================================================
# HÀM ĐIỀU PHỐI VÀ TRÍCH XUẤT ĐIỂM G-EVAL CHUYÊN SÂU
# =====================================================================
def run_geval_pipeline(df_data, num_samples=30):
    sources_list = df_data['source'].dropna().tolist()
    geval_metrics = ['Relevance', 'Coherence', 'Consistency', 'Fluency', 'Entity_Precision']
    score_accumulator = {name: {m: [] for m in geval_metrics} for name in model_columns}
    
    random.seed(42)
    sampled_indices = random.sample(range(len(sources_list)), min(num_samples, len(sources_list)))
    chunks = [model_columns[i:i + 4] for i in range(0, len(model_columns), 4)]
    
    print(f"🤖 Đang kích hoạt G-Eval (Trọng tài LLM Gemini Native) trên {len(sampled_indices)} mẫu ngẫu nhiên...")
    for idx in tqdm(sampled_indices, desc="[G-EVAL ENGINE]"):
        doc = sources_list[idx]
        for chunk in chunks:
            current_summaries = [df_data[name].iloc[idx] for name in chunk]
            while len(current_summaries) < 4: 
                current_summaries.append("Bản tóm tắt trống.")
                
            json_scores = call_geval_judge(doc, current_summaries)
            
            clean_json_scores = {}
            for k, v in json_scores.items():
                clean_json_scores[k.lower().strip()] = v

            for i, name in enumerate(chunk):
                for m in geval_metrics:
                    metric_key = m.lower() 
                    try:
                        score = clean_json_scores[metric_key][i]
                        score_accumulator[name][m].append(float(score))
                    except:
                        score_accumulator[name][m].append(50.0) 
        # Giới hạn điều phối an toàn tốc độ request tránh chạm RPM tầng Free
        time.sleep(4.0) 
            
    geval_report_rows = []
    for m in geval_metrics:
        row = {"Tiêu chí đánh giá (G-Eval Metrics)": m}
        for name in model_columns:
            row[name] = np.mean(score_accumulator[name][m])
        geval_report_rows.append(row)
        
    return pd.DataFrame(geval_report_rows)

# =====================================================================
# TIẾN TRÌNH THỰC THI CHÍNH, LƯU FILE VÀ HIỂN THỊ
# =====================================================================
df_geval_raw_results = run_geval_pipeline(df_master_saved, num_samples=30)
df_ieee_style = df_geval_raw_results.set_index("Tiêu chí đánh giá (G-Eval Metrics)")

df_ieee_style.to_csv('/kaggle/working/geval_metrics_summary_ieee.csv', index=True)
print("\n💾 ĐÃ LƯU FILE THÀNH CÔNG! Đường dẫn lưu trữ: /kaggle/working/geval_metrics_summary_ieee.csv")

def display_ultimate_geval_report(df_report):
    format_config = {col: "{:.2f}" for col in df_report.columns}
    def highlight_max_row(s):
        is_max = s == s.max()
        return ['color: #27ae60; font-weight: bold; background-color: #f2f9f5;' if v else '' for v in is_max]
    
    styled_html = (df_report.style
        .format(format_config)
        .apply(highlight_max_row, axis=1) 
        .set_caption("🏆 BẢNG TỔNG HỢP KẾT QUẢ ĐÁNH GIÁ ĐỊNH TÍNH CHUYÊN SÂU QUA G-EVAL (CHUẨN HÓA IEEE)")
        .set_table_styles([
            {'selector': 'caption', 'props': [('font-size', '16px'), ('font-weight', 'bold'), ('margin-bottom', '15px'), ('color', '#2c3e50'), ('text-align', 'center')]},
            {'selector': 'th', 'props': [('background-color', '#2c3e50'), ('color', 'white'), ('font-size', '11px'), ('padding', '10px'), ('text-align', 'center'), ('border', '1px solid #ddd')]},
            {'selector': 'td', 'props': [('font-size', '11px'), ('padding', '10px'), ('text-align', 'center'), ('border', '1px solid #ddd')]}
        ])
    )
    display(styled_html)

display_ultimate_geval_report(df_ieee_style)

🤖 Đang kích hoạt G-Eval (Trọng tài LLM Gemini Native) trên 30 mẫu ngẫu nhiên...


[G-EVAL ENGINE]: 100%|██████████| 30/30 [03:38<00:00,  7.29s/it]


💾 ĐÃ LƯU FILE THÀNH CÔNG! Đường dẫn lưu trữ: /kaggle/working/geval_metrics_summary_ieee.csv


,Baseline Transformer (Vanilla),Baseline Transformer (Penalty),Improved Baseline (Vanilla),Improved Baseline (Penalty),Soft Prompt Mamba (Vanilla),Soft Prompt Mamba (Penalty),Entity Gated Mamba (Vanilla),Entity Gated Mamba (Penalty),Transformer Hypersphere (Vanilla),Transformer Hypersphere (Penalty),Entity Gated Mamba With Label Smoothing (Vanilla),Entity Gated Mamba With Label Smoothing (Penalty),Entity Guided Hybrid Mamba (Vanilla),Entity Guided Hybrid Mamba (Penalty),Entity Guided Pure Transformer (Vanilla),Entity Guided Pure Transformer (Penalty)
Tiêu chí đánh giá (G-Eval Metrics),,,,,,,,,,,,,,,,
Relevance,20.33,22.83,22.33,23.17,23.67,24.33,24.50,24.33,42.50,41.67,26.83,26.50,38.33,39.33,44.83,46.67
Coherence,19.67,22.83,22.67,23.83,25.67,26.67,27.17,27.00,39.00,38.50,29.67,29.17,33.67,35.83,40.83,44.00
Consistency,21.33,22.50,22.00,22.17,22.67,22.83,23.17,23.00,40.67,40.67,26.00,25.50,36.67,37.17,41.33,45.00
Fluency,22.50,28.00,27.50,29.50,35.33,37.67,36.33,36.50,46.00,45.50,38.67,37.67,37.33,39.83,44.33,48.33
Entity_Precision,21.33,22.50,22.17,22.50,23.67,24.00,24.33,24.17,45.33,45.33,27.33,26.83,43.00,43.83,47.83,51.00
